# usb_led_controller

USB-C powered RGB LED controller: VBUS→LDO 3.3 V→STM32F042K6T6 MCU driving R/G/B LED channels via PWM + current-limit resistors.

In [ ]:
import hw_toolkit as hw

board = hw.Board("usb_led_controller")
board

In [ ]:
# USB-C connector — GCT USB4125-GF-A, 16-pin mid-mount USB-C
usbc_conn = board.module(
    id="usbc_conn",
    category="connector",
    mpn="USB4125-GF-A",
    package="USB-C-SMD-16P",
    price_usd=0.95,
    manufacturer="GCT",
)
usbc_conn

In [ ]:
# 3.3V LDO regulator — AP2112K-3.3TRG1, SOT-23-5
# Pins: VIN, VOUT, GND, EN (active-high, tie to VIN for always-on)
ldo = board.module(
    id="ldo",
    category="ldo_regulator",
    mpn="AP2112K-3.3TRG1",
    package="SOT-23-5",
    price_usd=0.35,
    manufacturer="Diodes Inc",
)
ldo

In [ ]:
# MCU — STM32F042K6T6, LQFP-32, USB FS capable
# Relevant pins: VDD, GND, USB_DP (PA12), USB_DM (PA11),
#                TIM1_CH1 (PA8), TIM1_CH2 (PA9), TIM1_CH3 (PA10)
mcu = board.module(
    id="mcu",
    category="microcontroller",
    mpn="STM32F042K6T6",
    package="LQFP-32",
    price_usd=2.10,
    manufacturer="STMicroelectronics",
)
mcu

In [ ]:
# RGB LED — LTST-C19HE1WT common-anode RGB, 0805
led = board.module(
    id="led",
    category="led",
    mpn="LTST-C19HE1WT",
    package="LED-0805",
    price_usd=0.25,
    manufacturer="Lite-On",
)
led

In [ ]:
# Current-limit resistors for R, G, B channels (33 Ω → ~20 mA at 3.3 V)
r_red   = board.resistor("R1", "33",  package="0603")
r_green = board.resistor("R2", "33",  package="0603")
r_blue  = board.resistor("R3", "33",  package="0603")
r_red, r_green, r_blue

In [ ]:
# Decoupling capacitors on MCU 3.3 V rail
c_bulk    = board.capacitor("C1", "4.7uF", package="0805")   # bulk
c_bypass1 = board.capacitor("C2", "100nF", package="0402")   # high-freq bypass
c_bypass2 = board.capacitor("C3", "100nF", package="0402")   # high-freq bypass
c_bulk, c_bypass1, c_bypass2

In [ ]:
# --- Power nets ---
v5   = board.power("v5",   voltage_v=5.0)   # VBUS from USB-C connector
v3v3 = board.power("v3v3", voltage_v=3.3)   # LDO output rail
gnd  = board.gnd()                           # common GND

# 5 V VBUS: connector → LDO input; EN tied to VIN (always-on)
v5 += "usbc_conn.VBUS", "ldo.VIN", "ldo.EN"

# 3.3 V: LDO output → MCU supply, decoupling caps, LED anode
v3v3 += "ldo.VOUT", "mcu.VDD", "c1.POS", "c2.POS", "c3.POS", "led.A"

# GND: all grounds
gnd += "usbc_conn.GND", "ldo.GND", "mcu.GND"
gnd += "c1.NEG", "c2.NEG", "c3.NEG"

gnd

In [ ]:
# --- USB data nets (manual, avoiding usbc() factory to keep full control) ---

# D+ / D- differential pair: connector → MCU USB FS pins
usb_dp, usb_dm = board.diff_pair("usb", protocol="usb")
usb_dp += "usbc_conn.DP",  "mcu.USB_DP"
usb_dm += "usbc_conn.DM",  "mcu.USB_DM"

# CC1/CC2: used for cable orientation detect (5.1k pull-down per spec)
# Connect between connector and ground via resistors modeled as signals
cc1 = board.signal("cc1", protocol="usb")
cc2 = board.signal("cc2", protocol="usb")
cc1 += "usbc_conn.CC1", "usbc_conn.CC1_RTERM"
cc2 += "usbc_conn.CC2", "usbc_conn.CC2_RTERM"

# SBU lines — intentionally unused, route to nc sentinels
nc_sbu1 = board.nc("nc_sbu1")
nc_sbu2 = board.nc("nc_sbu2")
nc_sbu1 += "usbc_conn.SBU1"
nc_sbu2 += "usbc_conn.SBU2"

usb_dp, usb_dm

In [ ]:
# --- PWM signal nets: MCU → resistors → LED cathodes ---
pwm_r = board.signal("pwm_r", protocol="pwm")
pwm_g = board.signal("pwm_g", protocol="pwm")
pwm_b = board.signal("pwm_b", protocol="pwm")

# MCU TIM1 channels drive resistor A-side
pwm_r += "mcu.TIM1_CH1", "r1.A"
pwm_g += "mcu.TIM1_CH2", "r2.A"
pwm_b += "mcu.TIM1_CH3", "r3.A"

# Resistor B-side → LED cathodes
led_r = board.signal("led_r", protocol="analog")
led_g = board.signal("led_g", protocol="analog")
led_b = board.signal("led_b", protocol="analog")

led_r += "r1.B", "led.KR"
led_g += "r2.B", "led.KG"
led_b += "r3.B", "led.KB"

pwm_r, pwm_g, pwm_b

In [ ]:
# --- Sanity summary ---
print(board.summary())

In [ ]:
# --- ERC check ---
board.check_erc(
    expected_codes=(
        "pin_not_connected",         # intentional NCs (SBU, etc.)
        "lib_symbol_issues",         # hwagent lib synthesized at runtime
        "pin_to_pin",                # rails tied directly to pins (LDO EN=VIN)
        "power_pin_not_driven",      # connector power pins without PWR_FLAG
        "unconnected_wire_endpoint", # synthesized wire-layout artifact
    )
)
print("ERC passed")

In [ ]:
# --- Export KiCad project zip ---
import pathlib

out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/usb_led_controller/usb_led_controller.zip")
board.export_kicad(out, unzip=True)
print(f"Exported: {out}")
print(f"Exists:   {out.exists()}")